# Introduction to Scikit-Learn (sklearn)

This notebook demonstrates some of the most useful functions of the
beautiful Scikit-Learn library.

What we're going to cover:

0. An end-to-end Scikit-Learn workflow
1. Getting the data ready
2. Choose the right estimator (model) / algorithm for our problems
3. Fit the model/algorithm and use it to make predictions on our data
4. Evaluating a model
5. Improve a model
6. Save and load a trained model
7. Putting it all together

## 0. An end-to-end Scikit-Learn workflow

In [ ]:
from xml.dom import ValidationErr

# 1. Get the data ready
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
from fontTools.ttx import process

heart_disease = pd.read_csv('./data/heart-disease.csv')
heart_disease

In [ ]:
# Create `X` (the "feature matrix")
# AKA data or feature variables
X = heart_disease.drop('target', axis=1)
X

In [ ]:
# Create the y (AKA labels or label matrix)
y = heart_disease['target']
y

## 2. Choose the right model and hyperparameters

In [ ]:
# Remember, "hyperparameters" are like "dials" we can use to (fine) tune our model
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier()

# We'll keep the default hyperparameters
clf.get_params()

## 3. Fit the model to the training data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
clf.fit(X_train, y_train);

In [ ]:
# Make a (faulty) prediction (using **incorrectly shaped** data)
try:
    y_broken_label = clf.predict(np.array([0, 2, 3, 4]))
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')

In [ ]:
# Make a (correct) prediction (using **correctly shaped** data)
# Note: we are still on step 3 or our workflow
y_label = clf.predict(X_test)

In [ ]:
y_preds = clf.predict(X_test)
y_preds

In [ ]:
y_test

## 4. Evaluate the model on the training data ...

In [ ]:
clf.score(X_train, y_train)

In [ ]:
# ... and on the test data
clf.score(X_test, y_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(classification_report(y_test, y_preds))  # clf.predict(X_test)

See the article, [Understanding a Classification Report](https://medium.com/@kohlishivam5522/understanding-a-classification-report-for-your-machine-learning-model-88815e2ce397),
for an explanation of this report.

In [ ]:
# Calculate the "confusion matrix" for test and predicted values
confusion_matrix(y_test, y_preds)

In [ ]:
accuracy_score(y_test, y_preds)

## 5. Improve a model

In [ ]:
# - Generate a one-time 128-bit secret for the seed.
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# **Copy and paste** this value as the **hard-coded** random number
# generator seed
rng = np.random.default_rng(seed=77708057215789171477656921825058000355)

In [ ]:
# Try different amount of `n_estimators`
for i in range(10, 100, 10):
    print(f'Trying model with {i} estimators...')
    # I use the `default_rng` with a specified seed to generate reproducible results.
    # The maximum integer I want is the largest unsigned 32-bit integer. (This value
    # is the largest value compatible with the argument to `rng.integers`.)
    clf = RandomForestClassifier(
        n_estimators=i,
        random_state=rng.integers(np.iinfo(np.uint32).max)).fit(X_train, y_train)
    print(f'Model accuracy on test set: {clf.score(X_test, y_test) * 100:.2f}%')
    print('')  # simply to separate runs

The maximum accuracy of the test set, 91.80%, occurs with 50 estimators.

Consequently, we can **improve** our model by using 50 estimators.

## 6. Save and load a trained model

In [ ]:
# We can save a model using `pickle`.
import pickle

In [ ]:
# The video calls `pickle.dump(clf, open('random_forest_model.pkl', 'wb'))`.
# This call results in a type warning similar to
# "Expected SupportsWrite but got BinaryIO".
# This situation is resolved by using `with` below.

with open('./random_forest_model_1.pkl', 'wb') as f:
    pickle.dump(clf, f)

In [ ]:
# What happens if we try to import the saved model.
with open('./random_forest_model_1.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

In [ ]:
loaded_model.score(X_test, y_test)

This result is the **same** as the last model we tried (90 estimators). **Hooray!**

In [ ]:
# Let's "listify" the contents
what_were_covering = [
    '0. An end-to-end Scikit-Learn workflow',
    '1. Getting the data ready',
    '2. Choose the right estimator/algorithm/model for your problem',
    '3. Fitting your chosen machine learning model to data and using it to make a prediction',
    '4. Evaluating a machine learning model',
    '5. Improving predictions through experimentation (hyperparameter tuning)',
    '6. Saving and loading a pre-trained model',
    '7. Putting it all together in a pipeline',
]

In [ ]:
what_were_covering

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# %matplotlib inline

## 1. Getting our data ready to be used with machine learning

Three main tasks to complete:

1. Split the data into features and labels (usually `X` and `y`)
2. Filling (AKA imputing) or disregarding missing values
3. Converting non-numerical values to numerical values (also called "feature encoding")

In [ ]:
heart_disease.head()

In [ ]:
# The last column is our data to be predicted so we drop it
# Remember that `axis=1` is the **column** axis
# (`axis=0` is the **row** axis)
X = heart_disease.drop('target', axis=1)
X.head()

In [ ]:
y = heart_disease['target']
y.head()

In [ ]:
# Split the features and labels into training and test splits
# We reserve 20% of our data for testing. This amount is "negotiable" for
# different problems.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
X.shape

In [ ]:
len(heart_disease)

In [ ]:
len(heart_disease) * 0.8

In [ ]:
242 + 61

In [ ]:
len(heart_disease)

### 1.1 Make sure all data is **numerical**

In [ ]:
car_sales = pd.read_csv('./data/car-sales-extended.csv')
car_sales.head()

In [ ]:
car_sales['Doors'].value_counts()

Notice that `car_sales['Doors']` is **both** numeric **and** categorical.

It is numeric because its values are integers. But it is categorical
because it's (mathematical) range is only a small subset of integers.

As a consequence of the small subset of values, we will treat this column
as a **categorical** column (see our encoding code later).

In [ ]:
len(car_sales)

In [ ]:
car_sales.dtypes

In [ ]:
# Split into `X` and `y`
X = car_sales.drop('Price', axis=1)
y = car_sales['Price']

# Split into training and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# Build machine learning model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor() ## Create our model
try:
    model.fit(X_train, y_train) ## Fit our model
    model.score(X_test, y_test) ## Score our model
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')
# Score

Remember, we **must** convert strings (objects) to **numbers**

In [ ]:
# Turn the (object) categories into numbers
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identify features by **column names**
categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot',  ## Name our transform
                                  one_hot,  ## The specific transformer
                                  categorical_features)],  ## Applies **only** to `categorical_features`
                                remainder='passthrough')  ## Pass all other columns **unchanged**
transformed_X = transformer.fit_transform(X)
transformed_X

In [ ]:
pd.DataFrame(transformed_X)

In [ ]:
# An alternative to one-hot encoding
dummies = pd.get_dummies(car_sales[['Make', 'Colour', 'Doors']])
dummies

Now that our data is all numeric (zeros and ones), let's refit the model

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales)
transformed_X

In [ ]:
# We start with a well-known seed to our random number generator.
# Note: we've already done this, but we are doing it again to
# "train our fingers."
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# **MANUALLY** (re-)seed the `np.random.default_rng` with the
# seed from the previous cell.
rng = np.random.default_rng(seed=22232115356560702892793270496072236204)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2)

model.fit(X_train, y_train)

In [ ]:
model.score(X_test, y_test)

In [ ]:
print(sklearn.__version__)

### 1.2 What if I find **missing** data?

1. Fill them with some value (AKA imputation)
2. Remove the samples with missing data altogether

Neither of these techniques is "perfect" or "recommended"
- Replacing "missing" data might introduce bias or "throw away" "significant" information
- Removing samples completely results in less total data to use

In [ ]:
# Import car sales with missing data
car_sales_missing = pd.read_csv('./data/car-sales-extended-missing-data.csv')
car_sales_missing.head()

In [ ]:
# Extract `X` and `y` (features and values)
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

In [ ]:
# Calling `.sum()` takes advantage of Python "coercion" of Boolean types
# That is, True is converted to 1 when summing and False is converted to 0.
car_sales_missing.isna().sum()

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales)
transformed_X

At this point in the video, Python reports an exception:
"ValueError: Input contains NaN"

Because I'm using `sklearn` version 1.5.x, I **do not** see this error.

But I'll pretend like I do.

In [ ]:
car_sales_missing

In [ ]:
car_sales_missing['Doors'].value_counts()

In [ ]:
car_sales_missing['Doors'].mode()

#### Option 1: Fill missing data with `pandas`

In [ ]:
# Fill the "Make" column
car_sales_missing['Make'] = car_sales_missing['Make'].fillna('missing')

# Fill the "Colour" column
car_sales_missing['Colour'] = car_sales_missing['Colour'].fillna('missing')

# Fill the "Odometer (KM)" column
car_sales_missing['Odometer (KM)'] = car_sales_missing['Odometer (KM)'].fillna(car_sales_missing['Odometer (KM)'].mean())

# Fill the "Doors" column
# A little tricky because this column is actually a **categorical** column.
# Because the (overwhelming) majority of cars have 4 doors, we will
# replace all missing values in the 'Doors' column with the value 4.
# (Because 4 is the most common value, we could replace the hard-coded
# value of 4 with `car_sales_missing['Doors'].mode()`
car_sales_missing['Doors'] = car_sales_missing['Doors'].fillna(4)

In [ ]:
# Check our `DataFrame` again
car_sales_missing.isna().sum()

In [ ]:
# Because 'Price' is our value column, we **do not** want to replace
# missing values with another value. Instead, we will **remove** all
# rows that are missing a value in the 'Price' column.
car_sales_missing = car_sales_missing.dropna(subset=['Price'])

In [ ]:
car_sales_missing.isna().sum()

In [ ]:
len(car_sales_missing)

In [ ]:
# Remember, one must **always** split data into `X` and `y`
# (features and labels) after **changing** the data.
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales_missing)
transformed_X

#### Option 2. Fill missing values with Scikit-Learn

In [ ]:
# Read data (as usual)
car_sales_missing = pd.read_csv('./data/car-sales-extended-missing-data.csv')
car_sales_missing.head()

In [ ]:
# Check for missing data
car_sales_missing.isna().sum()

In [ ]:
# Remove rows **without** labels
car_sales_missing = car_sales_missing.dropna(subset=['Price'])
car_sales_missing.isna().sum()

In [ ]:
# Split into features and labels
X = car_sales_missing.drop('Price', axis=1)
y = car_sales_missing['Price']

In [ ]:
X.isna().sum()

In [ ]:
# Fill missing values from Scikit-LearnA
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Fill categorical values with 'missing' and numerical values with `mean()`
categorical_imputer = SimpleImputer(strategy='constant', fill_value='missing')
door_imputer = SimpleImputer(strategy='constant', fill_value=4)
numeric_imputer = SimpleImputer(strategy='mean')

# Define columns
categorical_features = ['Make', 'Colour']
door_feature = ['Doors']  ## Because 'Doors' column is a special case
numeric_features = ['Odometer (KM)']

# Create an imputer (that is, something the fills missing data)
imputer = ColumnTransformer([
    ('categorical_features', categorical_imputer, categorical_features),
    ('door_feature', door_imputer, door_feature),
    ('numeric_features', numeric_imputer, numeric_features),
])

# (Finally) Transform the features
filled_X = imputer.fit_transform(X)
filled_X

In [ ]:
# Check our code as we have done previously (using `isna().sum()`
car_sales_missing = pd.DataFrame(filled_X,
                                 columns=['Make', 'Colour', 'Doors', 'Odometer (KM)'])
car_sales_missing

In [ ]:
car_sales_missing.isna().sum()

In [ ]:
# Let's try to convert our data to numbers
# Remember our first step in Getting Your Data Ready.
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

categorical_features = ['Make', 'Colour', 'Doors']
one_hot = OneHotEncoder()
transformer = ColumnTransformer([('one_hot', one_hot, categorical_features)],
                                remainder='passthrough')

transformed_X = transformer.fit_transform(car_sales_missing)
transformed_X

Now our data is

- All numeric
- Filled (no missing values)

Let's fit a model!

In [ ]:
# We start with a well-known seed to our random number generator.
# Note: we've already done this, but we are doing it again to
# "train our fingers."
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# **MANUALLY** (re-)seed the `np.random.default_rng` with the
# seed from the previous cell.
rng = np.random.default_rng(seed=124608693003265431754472593407374562997)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(transformed_X, y, test_size=0.2)

model = RandomForestRegressor()
model.fit(X_train, y_train)
model.score(X_test, y_test)

In [ ]:
len(car_sales_missing), len(car_sales)

**Note**: The 50 less values in the transformed data is because we
dropped the rows (50 total) with missing values in the
transformed data.

In [ ]:
what_were_covering

## 2. Choosing the right estimator/algorithm/model for your problem

Some things to note:

- The package, `sklearn`, refers to machine learning models and
  algorithms as _estimators_
  - A _classifier_ is one type of estimator
  - A _regressor_ is another type of estimator
- Classification problem - predicting a **category**
  - For example, heart disease or not heart disease
  - Sometimes you see the term, `clf`
    - A TLA for **classifier**
    - Used as a classification estimator
- Regression problem - predicting a **number**
  - For example, the selling price of a car

If you're

- Working on a machine learning problem
- Looking to use `sklearn`
- Unsure what model you should use

Use the `sklearn` [machine learning model map](https://scikit-learn.org/stable/machine_learning_map.html)


### 2.1 Picking a machine learning model for a regression problem

Let's use the California Housing Data Set

Daniel, where are you getting these ideas from?

The `sklearn` documentation on [datasets](https://scikit-learn.org/stable/datasets.html)

- [Toy datasets](https://scikit-learn.org/stable/datasets/toy_dataset.html)
- [Real World datasets](https://scikit-learn.org/stable/datasets/real_world.html)
- [Generated datasets](https://scikit-learn.org/stable/datasets/sample_generators.html)
- [Loading other datasets](https://scikit-learn.org/stable/datasets/loading_other_datasets.html)


In [ ]:
# Get the California Housing dataset
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
housing

In [ ]:
# Let's turn our data into a `pandas` `DataFrame`
# Remember that the **datat** is in `housing.data`
housing_df = pd.DataFrame(housing['data'],
                          columns=housing['feature_names'])
housing_df

In [ ]:
# We must add our target data to the data frame
housing_df[housing.target_names[0]] = housing.target
housing_df

In [ ]:
# Lets change the target column name to "target"
housing_df = housing_df.rename({'MedHouseVal': 'target'}, axis=1)
housing_df

In [ ]:
# Split data into features and targets (`X` and `y`)
X = housing_df.drop('target', axis=1)
y = housing_df['target']

In [ ]:
X

In [ ]:
y

Let's use our "full" set of steps

- Import algorithm
- Setup random seed
- Create the data

In [ ]:
# Import algorithm
from sklearn import linear_model

In [ ]:
# Set up repeatable random number generator part 1

# Calculate our new seed
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# Set up repeatable random number generator part 2

# Create a random number generator with the specified seed
rng = np.random.default_rng(seed=187614292792846192168046308353096246446)

In [ ]:
# Create the data
X = housing_df.drop('target', axis=1)
y = housing_df['target'] # median house price in $100k

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate and fit the model on the **training** set
model = linear_model.Ridge(alpha=0.5)
model.fit(X_train, y_train)

# Check the score of the model on the test set
model.score(X_test, y_test)

What if `Ridge` did not work or the score did not fit our needs?

Well, we could always try a different model....

Let's try an "ensemble model"!

An "ensemble" is a combination of smaller models to try to make
better predictions than just a single model.

The ensemble models for `sklearn` can be found [here](https://scikit-learn.org/stable/modules/ensemble.html)

In [ ]:
# Import the `RandomForestRegressor` model class from the ensemble module
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Calculate our new random seed
import secrets

seed = secrets.randbits(128)
seed

In [ ]:
# Use this seed to seed a "new" random number generator
rng = np.random.default_rng(seed=20717103177106046274233922339781200557)

In [ ]:
# Create the data
X = housing_df.drop('target', axis=1)
y = housing_df['target']

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Create, fit, and score our random forest model
model = RandomForestRegressor()
model.fit(X_train, y_train)
model.score(X_test, y_test)


### 2.2 Choosing an estimator for a classification problem

Let's go to the map... [Online machine learning model map](https://scikit-learn.org/stable/machine_learning_map.html)

In [ ]:
heart_disease = pd.read_csv('./data/heart-disease.csv')
heart_disease

Consulted the [Online machine learning model map]if (https://scikit-learn.org/stable/machine_learning_map.html):
It's recommendation: try a "LinearSVC" model.

In [ ]:
# Import the classifier (and other required libraries)
from sklearn.svm import LinearSVC

In [ ]:
# Generate a problem-specific seed
import secrets

seed = secrets.randbits(123)
seed

In [ ]:
# Use the seed for our random number generator
rng = np.random.default_rng(seed=4577550920016562965326478789717866486)

In [ ]:
# Split the data into features and target
X = heart_disease.drop('target', axis=1)
y = heart_disease['target']

# Further split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate our model
clf = LinearSVC() ## `clf` is a TLA for "classifier"
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

In [ ]:
heart_disease['target'].value_counts()

In [ ]:
X

K-neighbors classifier

The package, `sklearn.neighbors`, "...provides functionality for
unsupervised and supervised neighbors-based learning methods.

Unfortunately, after reading the documentation of

- `NearestNeighbors`
- `KDTree`
- `BallTree`

I was unclear how to understand its application to our problem.
So... back to the video

Remember, when we previously moved to `RandomForestRegressor`, we
saw an **improvement** in our model score. Hopefully, we will see
an improvement using a `RandomForestClassifier`.

An interesting note from the [documentation](https://scikit-learn.org/stable/modules/ensemble.html#random-forests)

> The purpose of these two sources of randomness is to decrease the variance
> of the forest estimator. Indeed, individual decision trees typically
> exhibit high variance and tend to overfit. The injected randomness in
> forests yield decision trees with somewhat decoupled prediction errors.
> By taking an average of those predictions, some errors can cancel out.
> Random forests achieve a reduced variance by combining diverse trees,
> sometimes at the cost of a slight increase in bias. In practice the
> variance reduction is often significant hence yielding an overall
> better model.



In [ ]:
# Import the classifier (and other required libraries)
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Generate a problem-specific seed
import secrets

seed = secrets.randbits(123)
seed

In [ ]:
# Use the seed for our random number generator
rng = np.random.default_rng(seed=6318142287573278828684754060508691815)
# Split the data into features and target
X = heart_disease.drop('target', axis=1)
y = heart_disease['target']

# Further split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate our model
clf = RandomForestClassifier()  ## `clf` is a TLA for "classifier"
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

A machine learning tidbit from Daniel

> If you have structured data - AKA tables of data frames - use
> ensemble methods.
>
> Why? Because it will perform pretty well.
>
> If there are patterns

Tidbit (or shortcut):

| If you have... | Then use |
| -------------- | -------- |
| Structured data | Ensemble methods |
| Unstructured data | Deep learning methods |

Data Examples

| Structured Data | Unstructured Data |
| --------------- | ----------------- |
| Data in a table like `heart_disease` | - Images |
|                                      | - Audio |
|                                      | - Text |


In [ ]:
heart_disease

Remember, a critical success factor in data science is

- **Reducing** your time between experiments

In [ ]:
what_were_covering

## 3. Fit the model/algorithm to our data and use it to make predictions

### 3.1. Fitting the model to the data

Common "aliases"

- `X` - features, feature variables, data
- `y` - labels, targets, target values

In [ ]:
# Import the classifier (and other required libraries)
from sklearn.ensemble import RandomForestClassifier
# Generate a problem-specific seed

In [ ]:
import secrets

seed = secrets.randbits(123)
seed

In [ ]:
# Use the seed for our random number generator
rng = np.random.default_rng(seed=2075748097716160523040509094081475903)

In [ ]:
# Split the data into features and target
X = heart_disease.drop('target', axis=1)
y = heart_disease['target']

# Further split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Instantiate our model
clf = RandomForestClassifier()  ## `clf` is a TLA for "classifier"

# Fit the data
clf.fit(X_train, y_train)

# Evaluate the `RandomForestClassifier`
clf.score(X_test, y_test)

In [ ]:
X.head()

In [ ]:
y.head(), y.tail()

### Random Forest model deep dive

See [Random Forest Resources](notes/complete-ai-ml-ds/Random%20Forest%20Resources.md)
for more information.

### 3.2 Make predictions using a machine learning model

Two main ways to make predictions

- `predict()`
- `predict_proba()`

In [ ]:
# Use a trained model to make predictions

# We'll start with a mistake

try:
    clf.predict(np.array([2, 7, 1, 7, 2, 8])) # This choice **does not** work
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')

In [ ]:
clf.predict(X_test)

In [ ]:
np.array([y_test]).shape, clf.predict(X_test).shape

In [ ]:
# To evaluate our model, compare **predictions** to the
# "true" labels.
y_predictions = clf.predict(X_test)
np.mean(y_predictions == y_test)

In [ ]:
# This value is the same as...
clf.score(X_test, y_test)

In [ ]:
# We're getting ahead of ourselves, but...
# (Another way to calculate our score)
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_predictions)

In [ ]:
what_were_covering

Make predictions using `predict_proba()`

In [ ]:
# Note that `predict_proba()` returns the probabilities of a
# classification label.
clf.predict_proba(X_test)

In [ ]:
clf.predict_proba(X_test[:5])

In [ ]:
# Lets use `predict()` on the same data...
clf.predict(X_test[:5])

In [ ]:
X_test[:5]

In [ ]:
# Notice
0.89 + 0.11

The `predict()` method can also be used for regression models.

In [ ]:
housing_df.head()

In [ ]:
# We'll practice our steps starting with our `import(s)`
from sklearn.ensemble import RandomForestRegressor

# Create a random number generator with a known seed
rng = np.random.default_rng(seed=42)

# Create the features and labels
X = housing_df.drop('target', axis=1)
y = housing_df['target']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=rng.integers(np.iinfo(np.uint32).max))

# Create, fit, and score the model
model = RandomForestRegressor(random_state=rng.integers(np.iinfo(np.uint32).max))
model.fit(X_train, y_train)
model.score(X_test, y_test)

# Make predictions
y_predictions = model.predict(X_test)

In [ ]:
# Look at predictions
y_predictions[:10]

In [ ]:
np.array(y_test[:10])

In [ ]:
type(y_predictions)

In [ ]:
len(y_predictions), len(y_test)

In [ ]:
# Compare predictions to "the truth" using "mean absolute error"
from sklearn.metrics import mean_absolute_error
mean_absolute_error(y_test, y_predictions)

What does this value mean?

Our average absolute error is about 0.33.

In [ ]:
y_predictions

In [ ]:
try:
    housing_df['predictions'] = y_predictions
    housing_df.head()
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')

How can we resolve this issue?

Stay tuned for the next video.

In [ ]:
what_were_covering

## 4. Evaluating a machine learning model

Three ways with `scikit-learn` models/algorithms/estimators

- Estimator's built-in `score()` method
- The `scoring` parameter
- Problem-specific metric functions

Extra-curricular reading [Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)

### 4.1 Evaluating a model with the `score` method

In [ ]:
# We will use `RandomForestClassifier` as our "scoring guinea pig"
from sklearn.ensemble import RandomForestClassifier

# Initialize our random number generator with a well-known
# seed (for reproducibility)
rng = np.random.default_rng(seed=42)

# Create features (`X`) and labels (`y`)
X = heart_disease.drop('target', axis=1)
y = heart_disease['target']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=rng.integers(np.iinfo(np.uint32).max)
)

# Create, fit, and score our model
clf = RandomForestClassifier(random_state=rng.integers(np.iinfo(np.uint32).max))
clf.fit(X_train, y_train)
clf.score(X_test, y_test)

In [ ]:
# Score the model

# Remember, the highest possible value of the `score()` method is 1.0;
# the lowest value of the `score()` method is 0.0.
clf.score(X_train, y_train)

In [ ]:
y_train

In [ ]:
# Score the model on the **test** data
clf.score(X_test, y_test)

Remember to always be skeptical of 100% scores

Homework: Score our regression problem

In [ ]:
# Use the `score()` method on a `RandomForestRegressor`.
from sklearn.ensemble import RandomForestRegressor

# Initialize a random number generator for training and testing
rng = np.random.default_rng(seed=42)

# Split data into features and labels
X = housing_df.drop('target', axis='columns')
y = housing_df['target']

# Split our features into training and testing portions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=rng.integers(np.iinfo(np.uint32).max)
)

# Create, fit, and score our regressor
regressor = RandomForestRegressor(random_state=rng.integers(np.iinfo(np.uint32).max))
regressor.fit(X_train, y_train)

In [ ]:
# The default `score()` evaluation metric is r ** 2 for
# regression algorithms. The maximum value is 1.0; the
# minimum value is 0.0
regressor.score(X_train, y_train)

In [ ]:
regressor.score(X_test, y_test)

I think it looks good, but what do I know? I'm the student.

In [ ]:
housing_df.head()

### 4.2 Evaluating a model using the `scoring` parameter

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
# Typical model setup for a classifier
from sklearn.ensemble import RandomForestClassifier

# Create a well-known random number generator for training and
# testing purposes
rng = np.random.default_rng(seed=42)

# Extract features (`X`) and labels (`y`)
X = heart_disease.drop('target', axis='columns')
y = heart_disease['target']

# Split features and labels into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=rng.integers(np.iinfo(np.uint32).max)
)

# Create and fit a model.
classifier = RandomForestClassifier(random_state=rng.integers(np.iinfo(np.uint32).max))
classifier.fit(X_train, y_train)

In [ ]:
clf.score(X_test, y_test)

In [ ]:
# This calculation takes **all** the data (features and labels)
# and neither the training and testing sets.
cross_val_score(classifier, X, y, cv=5)

In [ ]:
cross_val_score(classifier, X, y, cv=3)

In [ ]:
cross_val_score(classifier, X, y, cv=10)

In [ ]:
fit_10 = _

In [ ]:
fit_10.mean(), fit_10.std()

In [ ]:
try:
    fit_10.describe()
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')

In [ ]:
pd.DataFrame(fit_10).describe()

In [ ]:
# Initialize our random seed
# I'm actually uncertain if this step is necessary.
rng = np.random.default_rng(seed=42)

# Single training and test split score
clf_single_score = classifier.score(X_test, y_test)

# Calculate the mean of the 5-fold cross-validation score
clf_cross_val_score = np.mean(cross_val_score(classifier, X, y, cv=5))

# Compone the two scores
clf_single_score, clf_cross_val_score

In [ ]:
# Default scoring parameter of classifier == mean accuracy
help(clf.score)

In [ ]:
# Let's look at the `scoring` parameter
# By default, it is set to `None`
cross_val_score(clf, X, y, cv=5, scoring=None)

#### 4.2.1 Classification model evaluation metrics

1. Accuracy
2. Area under ROC curve
3. Confusion Matrix
4. Classification report

In [ ]:
heart_disease.head()

In [ ]:
# Import the required packages
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Initialize a random number generator with a know seed
rng = np.random.default_rng(seed=42)

# Extract features (`X`) and labels (`y`)
X = heart_disease.drop('target', axis='columns')
y = heart_disease['target']

# Split data into training and testing sets
# Because we are investigating the "accuracy" metric, we
# **do not need** to split our data
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=rng.integers(np.iinfo(np.uint32).max)
# )

# Create the model and fit it to our training data
classifier = RandomForestClassifier(random_state=rng.integers(np.iinfo(np.uint32).max))
cross_val_score = cross_val_score(classifier, X, y, cv=5)
cross_val_score

In [ ]:
np.mean(cross_val_score)

In [ ]:
print(f'Heart Disease Classifier Cross-Validated Accuracy: {np.mean(cross_val_score) * 100:.2f}')

**Area under the receiver operating characteristic curve (AUC/ROC)**

* Area under curve (AUC)
* ROC (receiver operating characteristic) curve

ROC curves compare a model's

- True positive rate (TPR) and the
- False positive rate (FPR)

Definitions

| When a...                              | Then the situation is a... |
|----------------------------------------|----------------------------|
| Model predicts 1 and actual value is 1 | True positive              |
| Model predicts 1 and actual value is 0 | False positive             |
| Model predicts 0 and actual value is 0 | True negative              |
| Model predicts 0 and actual value is 0 | False negative             |



In [ ]:
# Import ROC curve metric
from sklearn.metrics import roc_curve

# Create X_train, X_test, y_train, y_test
# Remember that we **did not** split the data previously
X_train, X_test, y_train, y_test = \
    train_test_split(X, y, test_size=0.2,
                     random_state=rng.integers(np.iinfo(np.uint32).max))

# Create and fit the classifier
classifier = RandomForestClassifier(random_state=rng.integers(np.iinfo(np.uint32).max))
classifier.fit(X_train, y_train)

# Make predictions with probabilities
y_probs = classifier.predict_proba(X_test)

In [ ]:
y_probs[:10]

In [ ]:
len(y_probs)

In [ ]:
# Remember that ROC compares true positives to false positives
# Slice the first column of every row
y_probs_positive = y_probs[:, 1] # Extract model positives
y_probs_positive[:10]

In [ ]:
# Calculate the FPR, the TPR, and thresholds
fpr, tpr, threshold = roc_curve(y_test, y_probs_positive)

In [ ]:
# Check the false positive rates
fpr

In [ ]:
# And the true positive rates
tpr

Unfortunately, `sklearn` **does not** contain a function to **plot**
the ROC curve. We'll need to wait to discover how to plot this curve.

In [ ]:
# Create a function for plotting ROC curves
import matplotlib.pyplot as plt

def plot_roc_curve(fpr, tpr):
    """Plot an ROC curve given the false positive rate and
    true positive rate of a model.

    fpr - A `numpy.ndarray` of false positive rates.
    tpr - A `numpy.ndarray` of true positive rates.
    """

    # Plot ROC curve
    plt.plot(fpr, tpr, color='orange', label='ROC curve')

    # Plot line with no predictive power (AKA a "baseline")
    plt.plot([0, 1], [0, 1], color='darkblue', linestyle='--', label='Guessing')

    # Customize the plot
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR)')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend()

    plt.show()

In [ ]:
plot_roc_curve(fpr, tpr)

In [ ]:
# Let's also look at the AUC score
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test, y_probs_positive)

In [ ]:
# Plot perfect ROC curve and AUC score
fpr, tpr, thresholds = roc_curve(y_test, y_test)
plot_roc_curve(fpr, tpr)

In [ ]:
# Similarly, the perfect AUC score is 1.0
roc_auc_score(y_test, y_test)

Our next evaluation tool, a **confusion matrix**

** Confusion Matrix **

A _confusion matrix_ is a quick way to compare

- The labels actually predicted by a model
- The labels it was supposed to predict

In other words, a _confusion matrix_ illustrates where the model
is confused.

In [ ]:
# Let's get "hands on"
from sklearn.metrics import confusion_matrix

# Create a confusion matrix by making some predictions...
y_predictions = classifier.predict(X_test)

# ...and then creating a confusion matrix for those predictions
confusion_matrix(y_test, y_predictions)


In [ ]:
# Visualize a confusion matrix with `pd.crosstab()`
pd.crosstab(y_test, y_predictions,
            rownames=['Actual Labels'],
            colnames=['Predicted Labels'])

In [ ]:
# Total all the labels and compare to our test set
25 + 7 + 3 + 26, len(X_test)

In [ ]:
# How to install a conda package into the current environment
# from **within** a Jupyter notebook
import sys
!conda install --yes seaborn

In [ ]:
# Make our confusion matrix more visual with `seaborn`'s `heatmap()`
import seaborn as sns

# Set the font scale
sns.set(font_scale=1.5)

# Create a confusion matrix
conf_mat = confusion_matrix(y_test, y_predictions)

# Plot confusion matrix using heatmap
sns.heatmap(conf_mat)

plt.show()

** Confusion Matrix **

The next tool for evaluating a classification model is a
"confusion matrix."

A "confusion matrix" is a quick way to compare the

- Labels predicted by a model to
- The actual labels it was supposed to predict

In essence, this technique gives you an idea of where the model
is getting "confused."

In [ ]:
# See [here](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
# for sklean confusion matrix.
from sklearn.metrics import confusion_matrix

y_predictions = classifier.predict(X_test)

# Remember, `y_test` captures the **actual** results and
# `y_predictions` captures the **model predictions**.
confusion_matrix(y_test, y_predictions)

In [ ]:
pd.crosstab(y_test, y_predictions,
            rownames=['Actual Labels'],
            colnames=['Predicted Labels'])

### Creating a confusion matrix using Scikit-Learn

To use the new methods (at the time of recording) of creating
a confusion matrix with Scikit-Learn, you will need `sklearn`
version 1.0 or greater.

In [ ]:
sklearn.__version__

Newer versions (> 1.0) introduce two new methdos (functions?)

- `ConfusionMatrixDisplay.from_estimator()`
- `ConfusionMatrixDisplay.from_predictions()`

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(estimator=classifier, X=X, y=y)

plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_true=y_test,
    y_pred=y_predictions,
)

plt.show()

**Classification Report**

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_predictions))

# Imagine a situation in which

- We test 10,000 people for a diseases
- But only 1 person actually has it

This situation is where precision and recall become valuable. In fact,
in this situation, **all** the metrics become valuable.

In [ ]:
# Where precision and recall become valuable
disease_true = np.zeros(10000)
disease_true[0] = True # A 1

# Our model predicts 0 (False) for **every** case
disease_predictions = np.zeros(10000)

pd.DataFrame(classification_report(disease_true,
                                   disease_predictions,
                                   output_dict=True))

### 4.2.2 Regression model evaluation metrics

[Regression model evaluation metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)

We're going to cover

1. R^2 (pronounced R-squared) or coefficient of determination
2. Mean absolute error (MAE)
3. Mean squared error (MSE)

"In statistics, the **coefficient of determination**,
denoted by ${R}^2$ or ${r}^2$, ...is the proportion of the variation in the
dependent variable that is predictable from the independent variable.

"...The coefficient of determination normally ranges from 0 to 1.

"There are cases where ${R}^2$ can yield negative values."

(See [Coefficient of Determination)[https://en.wikipedia.org/wiki/Coefficient_of_determination].)

From (a later) video,

R^2 compares

- Model predictions to
- The target **mean**

The range of R^2 values is [$-\infty$, 1]. This range is characterized by:

A value of:
- $-infty$ indicates a very poor match
- 0 indicates that the predicted values are equal to the target mean
- 1 indicates that the predicted values perfectly predict the target values

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(seed=42)

X = housing_df.drop('target', axis='columns')
y = housing_df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

model = RandomForestRegressor(
    random_state=rng.integers(np.iinfo(np.uint32).max)
)
model.fit(X_train, y_train)

In [ ]:
model.score(X_test, y_test)

In [ ]:
housing_df

In [ ]:
y_test

In [ ]:
y_test.mean()

In [ ]:
from sklearn.metrics import r2_score

# Fill an array with `y_test.mean()`
y_test_mean = np.full(len(y_test), y_test.mean())

In [ ]:
y_test_mean[:10]

In [ ]:
# Since the R^2 score evaluates the difference from the mean,
# we expect a score of 0.0
r2_score(y_true=y_test, y_pred=y_test_mean)

In [ ]:
# Let's evaluate a "perfect" model
r2_score(y_true=y_test, y_pred=y_test)

In [ ]:
from sklearn.metrics import mean_absolute_error

y_predictions = model.predict(X_test)
mean_absolute_error(y_true=y_test, y_pred=y_predictions)

In [ ]:
from sklearn.metrics import mean_squared_error

y_predictions = model.predict(X_test)
mean_squared_error(y_true=y_test, y_pred=y_predictions)

**Mean absolute error (MAE)**

MAE is the average of the absolute error between predicted values
and actual values. It gives you an idea of how wrong are your
model predictions.

In [ ]:
# MAE
from sklearn.metrics import mean_absolute_error

y_predictions = model.predict(X_test)

mae = mean_absolute_error(y_true=y_test, y_pred=y_predictions)
mae

In [ ]:
y_predictions[:10]

In [ ]:
df = pd.DataFrame(data={'actual values': y_test,
                        'predicted values': y_predictions})
df['differences'] = df['predicted values'] - df['actual values']
df.head(10)

In [ ]:
df['differences'].mean()

In [ ]:
# MAE using formulas and differences
np.abs(df['differences']).mean()

In [ ]:
np.square(df['differences']).mean()

**Mean squared error (MSE)**

MSE is the mean of the square of the differences between actual
and predicted values.

In [ ]:
# Mean squared error
from sklearn.metrics import mean_squared_error

y_predictions = model.predict(X_test)

mse = mean_squared_error(y_true=y_test, y_pred=y_predictions)
mse

In [ ]:
# Let's investigate why MSE is **less than** MAE.
df['squared_differences'] = np.square(df['differences'])
df.head(10)

Our "interesting" observation

- Values between -1 and 1 become **smaller**
- Values less than -1 or greater that 1 become **larger**

In [ ]:
# Calculate MSE "by hand"
squared = np.square(df['differences'])
squared.mean()

In [ ]:
squared.mean() - mse

In [ ]:
# Let's investigate the effect of a "large" error
df_large_error = df.copy()
df_large_error.iloc[0]['squared_differences'] = 16
df_large_error

In [ ]:
# Calculate MSE on our copy with the large error
mse_large_error = df_large_error['squared_differences'].mean()
mse_large_error

In [ ]:
mse_large_error, mse

In [ ]:
mse_large_error - mse

In [ ]:
df_large_errors = df_large_error.copy()
df_large_errors[1:100] = 20
df_large_errors

In [ ]:
mse_large_errors = df_large_errors['squared_differences'].mean()
mse_large_errors

In [ ]:
mse_large_errors - mse

### 4.2.3 Finally using the `scoring` parameter

In [ ]:
from sklearn.model_selection import cross_val_score

# We'll start with classification
from sklearn.ensemble import RandomForestClassifier

# Set up random number generator
rng = np.random.default_rng(seed=42)

# Identify features and labels
X = heart_disease.drop('target', axis='columns')
y = heart_disease['target']

classifier = RandomForestClassifier(
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

In [ ]:
# Cross-validation accuracy
#
# If `scoring` parameter is `None`, the default scoring evaluation
# metric of the estimator is used. For classification models, the
# default scoring evaluation metric is `accuracy'.
cv_acc = cross_val_score(
    classifier, X, y, cv=5, scoring=None,
)
cv_acc

In [ ]:
# Cross-validated accuracy
print(f'The cross-validated accuracy is {np.mean(cv_acc) * 100:.2f}%')

In [ ]:
# An alternative
# Remember, `accuracy` is the **default** scoring method
cv_acc = cross_val_score(
    classifier, X, y, cv=5, scoring='accuracy'
)
cv_acc

In [ ]:
# Cross-validated accuracy
print(f'The cross-validated accuracy is {np.mean(cv_acc) * 100:.2f}%')

In [ ]:
# Let's try another scorning metric: `precision`
cv_precision = cross_val_score(
    classifier, X, y, cv=5, scoring='precision',
)
cv_precision

In [ ]:
print(f'The cross-validated precision is {np.mean(cv_precision)}')

In [ ]:
# Let's use the 'recall' metric for scoring
cv_recall = cross_val_score(
    classifier, X, y, cv=5, scoring='recall',
)
cv_recall

In [ ]:
print(f'The cross-validate recall is {np.mean(cv_recall)}')

In [ ]:
cv_f1 = cross_val_score(
    classifier, X, y, cv=5, scoring='f1'
)
cv_f1

In [ ]:
print(f'The cross-validated f1 score is {np.mean(cv_f1)}')

Let's see the scoring parameter being used for a regression problem

In [ ]:
# Set up a regression problem for scaring
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(seed=42)

X = housing_df.drop('target', axis='columns')
y = housing_df['target']

regressor = RandomForestRegressor(
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

In [ ]:
try:
    cv_r2 = cross_val_score(
        regressor, X, y, cv=5, scoring=mae,
    )
except Exception as e:
    print(f'Exception: {type(e).__name__}: {e}')

In [ ]:
cv_mae = cross_val_score(
    regressor, X, y, cv=5, scoring='neg_mean_absolute_error',
)
np.mean(cv_mae)

In [ ]:
cv_mae

In [ ]:
cv_mse = cross_val_score(
    regressor, X, y, cv=5, scoring='neg_mean_squared_error'
)
np.mean(cv_mse)

In [ ]:
cv_mse

## 4.3 Using different evaluation metrics as Scikit-Learn functions

The third way to evaluate `scikit-learn` machine learning
models/estimators is to use the `scikit-learn.metrics` module.

Use `sklearn` functions to evaluate a classification.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(seed=42)

# Create X and y
X = heart_disease.drop('target', axis=1)
y = heart_disease['target']

# Split the data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

# Create and fit a classifier
classifier = RandomForestClassifier(random_state=rng.integers(np.iinfo(np.uint32).max))
classifier.fit(X_train, y_train)

# Make predictions
y_predictions = classifier.predict(X_test)

In [ ]:
# Score using `accuracy_score()`
clf_accuracy = accuracy_score(y_true=y_test, y_pred=y_predictions)
clf_accuracy

In [ ]:
# Score using `precision_score()`
clf_precision = precision_score(y_true=y_test, y_pred=y_predictions)
clf_precision

In [ ]:
# Score using `recall_score()`
clf_recall = recall_score(y_true=y_test, y_pred=y_predictions)
clf_recall

In [ ]:
# Score using `f1_score()`
clf_f1 = f1_score(y_true=y_test, y_pred=y_predictions)
clf_f1

Use `sklearn` function to evaluate a regression.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Split into features and labels
X = housing_df.drop('target', axis=1)
y = housing_df['target']

# Initialize our random number generator for reproducibility
rng = np.random.default_rng(seed=42)

# Split into training and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

# Create and fit the model
regressor = RandomForestRegressor(
    random_state=rng.integers(np.iinfo(np.uint32).max)
)
regressor.fit(X_train, y_train)

# Make predictions
y_predictions = regressor.predict(X_test)

In [ ]:
# Score using `r2_score()`
model_r2 = r2_score(y_true=y_test, y_pred=y_predictions)
model_r2

In [ ]:
# Score using `mean_absolute_error()`
model_mae = mean_absolute_error(y_true=y_test, y_pred=y_predictions)
model_mae

In [ ]:
# Score using `mean_squared_error()`
model_mse = mean_squared_error(y_true=y_test, y_pred=y_predictions)
model_mse

In [ ]:
what_were_covering


## 5. Improving a model

Terminology
- First predictions == baseline predictions
- First model == baseline model

From a data perspective:
- Could we **collect** more data? (In general, the more data, the better.)
- Could we **improve** our data?

From a model perspective
- Is there a better model we could use
    - [Local machine learning model map](./ml_map.svg)
    - [Online machine learning model map](https://scikit-learn.org/stable/machine_learning_map.html)
- Can we improve **the current model**

General model improvements
- Use a simple model
    - For example, use a Linear SVC model
- An alternative to simple models are ensemble models

We have two different "levers" to improve our models
- Parameters
    - Models find these patterns in the data
- Hyperparameters
    - Settings on a model that one can (potentially) adjust to find (better) patterns

Three ways to adjust hyperparameters
1. By hand
2. Randomly with `RandomSearchCV`
3. Exhaustively with `GridSerchCV`



In [ ]:
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(seed=42)

clf = RandomForestClassifier(random_state=rng.integers(np.iinfo(np.uint32).max))

In [ ]:
clf.get_params()

### 5.1 Tuning hyperparameter by hand

Let's make three sets of data
- Training
- Validation
- Test

In [ ]:
# Let's remind ourselves of our baseline parameters
clf.get_params()

In [ ]:
clf.get_params()['n_estimators']

We will try to adjust:
- `max_depth`
- `max_features`
- `min_samples_leaf`
- `min_samples_split`
- `n_estimators`

In [ ]:
# Let's create an evaluation function so that we can evaluate
# all our models consistently
def evaluate_predictions(y_true, y_preds):
    """
    Performs an evaluation comparison on `y_true` labels
    versus `y_preds` (predicted) labels on a **classification model**3n

    :param y_true: The true labels for our classifier.
    :param y_preds: The predicted labels for our classifier.
    :return: result; A dictionary with the accuracy, precision,
    recall, and f1 scores
    """
    accuracy = accuracy_score(y_true, y_preds)
    precision = precision_score(y_true, y_preds)
    recall = recall_score(y_true, y_preds)
    f1 = f1_score(y_true, y_preds)

    result = {
        'accuracy': round(accuracy, 2),
        'precision': round(precision, 2),
        'recall': round(recall, 2),
        'f1': round(f1, 2),
    }

    print(f'Accuracy: {result['accuracy'] * 100:.2f}')
    print(f'Precision: {result['precision']:.2f}')
    print(f'Recall: {result['recall']:.2f}')
    print(f'F1: {result['f1']:.2f}')

    return result

In [ ]:
# This cell illustrates my solution to splitting the data into
# training, validation, and testing splits.
from sklearn.ensemble import RandomForestClassifier
rng = np.random.default_rng(seed=42)

X = heart_disease.drop('target', axis=1)
y = heart_disease['target']

X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.3,
    random_state=rng.integers(np.iinfo(np.uint32).max)
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5,
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

classifier = RandomForestClassifier(
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

In [ ]:
X_train.shape, X_validation.shape, X_test.shape

In [ ]:
# Fit and make predictions
classifier.fit(X_train, y_train)

# Make **baseline** predictions
y_preds = classifier.predict(X_validation)

baseline_metrics = evaluate_predictions(y_true=y_validation, y_preds=y_preds)
baseline_metrics

In [ ]:
# This call illustrates the solution used by the instructor in the class.
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(seed=42)

# Shuffle the data
heart_disease_shuffled = heart_disease.sample(
    frac=1.0,
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

X = heart_disease_shuffled.drop('target', axis=1)
y = heart_disease_shuffled['target']

# Split the data into training, validation and testing sets

# Training split is 705 of the data
train_split_count = round(0.7 * len(heart_disease_shuffled))
# Validation split is 15% of the data
valid_split_count = train_split_count + round(0.15 * len(heart_disease_shuffled))

X_train = X[:train_split_count]
y_train = y[:train_split_count]
X_validate = X[train_split_count:valid_split_count]
y_validate = y[train_split_count:valid_split_count]

# "The rest" works because we previously used `round()`
X_test = X[valid_split_count:]
y_test = y[valid_split_count:]

classifier = RandomForestClassifier(
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

In [ ]:
# Use the baseline parameters for a baseline prediction
clf.get_params()

In [ ]:
# Fit and make predictions
classifier.fit(X_train, y_train)

# Make **baseline** predictions
y_preds = classifier.predict(X_validation)

baseline_metrics = evaluate_predictions(y_true=y_validation, y_preds=y_preds)
baseline_metrics

Let's try to improve our results'

In [ ]:
rng = np.random.default_rng(seed=42)
classifier_2 = RandomForestClassifier(
    n_estimators=200,
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

classifier_2.fit(X_train, y_train)
y_preds_2 = classifier_2.predict(X_validation)

baseline_metrics_2 = evaluate_predictions(y_true=y_validation, y_preds=y_preds_2)
baseline_metrics_2

In [ ]:
# rng = np.random.default_rng(seed=42)

# classifier_3 = RandomForestClassifier(
#     max_depth=20,
# )

### 5.2 Hyperparameter tuning with `RandomizedSearchCV`

In [ ]:
    # NOTE: In scikit-learn version 1.1, the default value of
    # `max_features` # was changed from 'auto' to 'sqrt'. From
    # the error message that occurred when I included 'auto',
    # I think 'auto' has been removed.
    # 'max_features': ['auto', 'sqrt'],


In [ ]:
# Import the tool
from sklearn.model_selection import RandomizedSearchCV

# Create a grid of hyperparameters we wish to adjust
grid = {
    'n_estimators': [10, 100, 200, 500, 1000, 1200],
    'max_depth': [None, 5, 10, 20, 30],
    # NOTE: In scikit-learn version 1.1, the default value of
    # `max_features` was changed from 'auto' to 'sqrt'. From
    # the error message that occurred when I included 'auto',
    # I think 'auto' has been removed.
    # 'max_features': ['auto', 'sqrt'],
    'max_features': ['sqrt', 'log2', None],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 4],
}

rng = np.random.default_rng(seed=42)

# Split into X and y
X = heart_disease_shuffled.drop('target', axis=1)
y = heart_disease_shuffled['target']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

# Create the classifier
classifier = RandomForestClassifier(
    random_state=rng.integers(np.iinfo(np.uint32).max),
    n_jobs=1,
)

# Set up `RandomizedSearchCV`
# NOTE: because we are using cross-validation (`cv=5`)', we need
# **not** create a validation split.
rs_classifier = RandomizedSearchCV(
    estimator=classifier,
    param_distributions=grid,
    n_iter=10, # number of models to try
    cv=5,
    verbose=2
)

# Fit the `RandomizedSearchCV` version of `classifier`
rs_classifier.fit(X_train, y_train)

In [ ]:
rs_classifier.best_params_

Once we have found the best hyper-parameters, the `predict()`
method will simply use them.

In [ ]:
# Make predictions with the best hyperparameters
rs_y_preds = rs_classifier.predict(X_test)

# Evaluate the predictions
rs_metrics = evaluate_predictions(y_test, rs_y_preds)

### 5.3 Hyperparameter tuning using `GridSearchCV`

In [ ]:
# We've already calculated a grid of hyper-parameters and vales
grid

In [ ]:
6 * 5 * 2 * 3 * 3 * 5

In [ ]:
# Let's create a reduced grid for our grid search

# Start with our previous `grid` value
grid_2 = {
    'n_estimators': [200, 500, 1000],
    'max_depth': [20, 30],
    'max_features': ['sqrt', 'log2'],
    'min_samples_split': [6],
    'min_samples_leaf': [1, 2],
}

In [ ]:
# Now our search space, including 5-fold cross-validation, is
3 * 2 * 2 * 1 * 2 * 5

In [ ]:
# Our grid search
from sklearn.model_selection import GridSearchCV, train_test_split

rng = np.random.default_rng(seed=42)

# Split into X and y
X = heart_disease_shuffled.drop('target', axis=1)
y = heart_disease_shuffled['target']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=rng.integers(np.iinfo(np.uint32).max)
)

# Create the classifier
classifier = RandomForestClassifier(
    random_state=rng.integers(np.iinfo(np.uint32).max),
    n_jobs=1,
)

# Set up `GridSearchCV`
# NOTE: because we are using cross-validation (`cv=5`)', we need
# **not** create a validation split.
gs_classifier = GridSearchCV(
    estimator=classifier,
    param_grid=grid_2,
    cv=5,
    verbose=2
)

# Fit the `GridSearchCV` version of `classifier`
gs_classifier.fit(X_train, y_train)

In [ ]:
gs_classifier.best_params_

In [ ]:
# Time to test our predictions
gs_y_preds = gs_classifier.predict(X_test)

gs_metrics = evaluate_predictions(y_test, gs_y_preds)

After our three-step process

1. Try different hyperparameters by hand
2. Use `RandomizedSearchCV` to refine hand results
3. Use `GridSearchCV` to further refine randomized search results
4. Iterate steps 1-3 as needed
5. Compare our different models

In [ ]:
compare_results = pd.DataFrame({
    'baseline': baseline_metrics,
    'classifier_2': baseline_metrics_2,
    'random_search': rs_metrics,
    'grid_search': gs_metrics,
})
compare_results.plot.bar(figsize=(10, 8))
plt.show()



In [ ]:
what_were_covering

## 6. Saving and Loading a Model

Two ways to perform these actions

1. Using Python's `pickle' module
2. Using the `joblib` module

In [ ]:
# Let's use `pickle`
import pickle

# Save an existing model to a file
with open('gs_random_forest_model_1.pkl', 'wb') as f:
    pickle.dump(gs_classifier, f)

In [ ]:
# Load a pickled model
with open('gs_random_forest_model_1.pkl', 'rb') as f:
    loaded_pickle_model = pickle.load(f)

In [ ]:
# Check the loaded model by making some predictions
pickle_y_predictions = loaded_pickle_model.predict(X_test)
evaluate_predictions(y_test, pickle_y_predictions)

In [ ]:
gs_metrics = evaluate_predictions(y_test, gs_y_preds)